# Parte III - Busca local com pontos de coleta

Analise de Hill Climbing e Simulated Annealing para otimizar a ordem de visita dos pontos `C`.

In [ ]:
from pathlib import Path
import ast
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

BASE = Path.cwd()
if BASE.name == "notebooks":
    BASE = BASE.parent
DATA = BASE / "datasets"

arquivos = {
    "Hill Climbing": "resultados_hill_climbing.csv",
    "Simulated Annealing": "resultados_simulated_annealing.csv",
}

frames = []
for algoritmo, arquivo in arquivos.items():
    caminho = DATA / arquivo
    if caminho.exists():
        df = pd.read_csv(caminho)
        df["algoritmo"] = algoritmo
        frames.append(df)

local = pd.concat(frames, ignore_index=True)
mazes = pd.read_csv(DATA / "mazes.csv").rename(columns={"id": "maze_id"})
local = local.merge(mazes, on="maze_id", how="left")
local.head()

In [ ]:
# Metricas obrigatorias: melhor, pior, media, tempo medio, iteracoes medias e taxa de sucesso.
def taxa_sucesso_5pct(grupo):
    limite = grupo["custo"].min() * 1.05
    return (grupo["custo"] <= limite).mean() * 100

resumo = (
    local.groupby("algoritmo")
    .agg(
        execucoes=("maze_id", "count"),
        melhor_custo=("custo", "min"),
        pior_custo=("custo", "max"),
        custo_medio=("custo", "mean"),
        tempo_medio=("tempo_segundos", "mean"),
        iteracoes_medias=("iteracoes", "mean"),
    )
)
resumo["taxa_sucesso_5pct"] = local.groupby("algoritmo").apply(taxa_sucesso_5pct, include_groups=False)
resumo.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.boxplot(data=local, x="algoritmo", y="custo", ax=axes[0])
axes[0].set_title("Distribuicao dos custos")
axes[0].set_xlabel("")

sns.barplot(data=resumo.reset_index(), x="algoritmo", y="tempo_medio", ax=axes[1])
axes[1].set_title("Tempo medio")
axes[1].set_xlabel("")

sns.barplot(data=resumo.reset_index(), x="algoritmo", y="iteracoes_medias", ax=axes[2])
axes[2].set_title("Iteracoes medias")
axes[2].set_xlabel("")

for ax in axes:
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# Curva de convergencia: usa a melhor execucao de cada algoritmo.
def parse_curva(valor):
    try:
        return ast.literal_eval(valor)
    except Exception:
        return []

melhores = local.loc[local.groupby("algoritmo")["custo"].idxmin()].copy()
curvas = []
for _, row in melhores.iterrows():
    for iteracao, custo in parse_curva(row.get("curva_convergencia", "[]")):
        curvas.append({"algoritmo": row["algoritmo"], "iteracao": iteracao, "custo": custo})

df_curvas = pd.DataFrame(curvas)
plt.figure(figsize=(10, 5))
sns.lineplot(data=df_curvas, x="iteracao", y="custo", hue="algoritmo", marker="o")
plt.title("Iteracao x melhor custo")
plt.tight_layout()
plt.show()

## Questoes de analise solicitadas

1. Hill-Climbing ficou preso em minimo local? Use a variacao entre melhor/pior custo e a curva de convergencia para justificar.
2. Simulated Annealing encontrou solucoes melhores? Compare melhor custo, custo medio e distribuicao dos custos.
3. Como a temperatura inicial influenciou os resultados? **Dado faltante:** o CSV atual nao salva `temp_inicial`.
4. Como a taxa de resfriamento afetou a convergencia? **Dado faltante:** o CSV atual nao salva `taxa_resfriamento`.
5. A busca local encontrou sempre a solucao otima? Nao ha garantia teorica. Para afirmar empiricamente, seria preciso comparar com uma busca exaustiva para mapas pequenos.
6. Qual foi o compromisso entre tempo e qualidade? Compare custo medio, melhor custo e tempo medio nas tabelas/graficos acima.